In [ ]:
import h5py
import numpy as np
from time import time


import os
import sys
sys.path.append(os.path.join(os.path.dirname(__file__), "../src/engine"))

from turboquant import unpack_tq_dequant_bits
from rabitq import unpack_all_data_rb
from evp import de_evp, unpack_evp_dequant_bits_llvm, unpack_all_data, unpack_batch_simd
from utils import bitpack_q_data_jit
from paper_experiments.quantisation_functions import make_turboquant_data, make_evp_data, make_rabit_data

In [2]:
# Globals
D_SIZE = 1_000_000
QUERY_ID = 0
NANO_SECODNS = 1e9

In [3]:
with h5py.File("/data/kb267/mf_dino2.h5") as f:
    data = f["embeddings"][:D_SIZE].astype(np.float32)

query = data[QUERY_ID]
DIMS = query.shape[0]
print(DIMS)
print(data.shape[0])

384
1000000


In [4]:
start = time()

# Quantise the data
tq_data, rrot = make_turboquant_data(data)
print("tq complete")

end = time() - start
print(f'time: {end}s')

tq complete
time: 24.331819534301758s


In [5]:
start = time()

evp_data = make_evp_data(data)
print("evp complete")

end = time() - start
print(f'time: {end}s')

evp complete
time: 5.4908857345581055s


In [6]:
start = time()
rabit_data, mags = make_rabit_data(data)
print("rabit complete")
end = time() - start
print(f'time: {end}s')

rabit complete
time: 192.99385976791382s


In [7]:
# Bit pack the quantisations
evp_bitpacked = bitpack_q_data_jit(evp_data)
tq_bitpacked = bitpack_q_data_jit(tq_data)
rabit_bitpacked = bitpack_q_data_jit(rabit_data)

In [8]:
print(evp_bitpacked[2])
print(rabit_bitpacked[2])
print(tq_bitpacked[2])

[ 1897424012435854593  6129708357451848228 10765051531407804549
  9378291397000992410  5084869928701766225  4726054306373456486
  5337205725374792261  6370711217713079826  9585226918581830032
  6243280479626221649  1844806880180130900  9516485727763321445]
[ 1916587486513616214  7284125203509877609 11144849601334036889
 10837462418991146666  5085239433396612770  5015482411817723494
 11125457512599017290  7614009571713117014 15355768780834100645
  6554052034211175846  6533146744670479769 11030910442418034298]
[ 2414674846927127037 17482545128574178443  5939072492285524646
  7365237098293528170  3127344609189342842  1719835586459040411
 11939735453287144103  3209207647401450970   391082156050161209
 15516824210296104568  6731375822091416235  7395365665719614020]


In [9]:
query = np.ascontiguousarray(query, dtype=np.float32)
evp_test = np.ascontiguousarray(evp_bitpacked[QUERY_ID], dtype=np.uint64)

start = time()

res = unpack_evp_dequant_bits_llvm(query, evp_test, DIMS)

end = time() - start

In [10]:
print(f'time: {end}s')

print(res)

time: 0.12767529487609863s
0.8976272723268522


In [22]:
start = time()

res = unpack_batch_simd(query, evp_bitpacked, DIMS)

end = time() - start

In [23]:
print(f'time: {end}s')
print(f'time/v: {end/D_SIZE * NANO_SECODNS} ns')

print(res)

time: 0.47113823890686035s
time/v: 471.13823890686035 ns
[0.89762727 0.05176839 0.04791404 ... 0.02319447 0.03668125 0.1086978 ]


In [24]:
start = time()

res = unpack_all_data(query, evp_bitpacked, DIMS)

end = time() - start

In [25]:
print(f'time: {end}s')
print(f'time/v: {end/D_SIZE * NANO_SECODNS} ns')

print(res)

time: 0.4300730228424072s
time/v: 430.0730228424072 ns
[0.8976273  0.05176839 0.04791404 ... 0.02319447 0.03668125 0.10869779]


In [31]:
start = time()

res = unpack_all_data_rb(query, rabit_bitpacked, DIMS, mags)

end = time() - start

In [32]:
print(f'time: {end}s')
print(f'time/v: {end/D_SIZE * NANO_SECODNS} ns')

print(res)

time: 0.43422389030456543s
time/v: 434.22389030456543 ns
[0.9184504  0.03195373 0.0202792  ... 0.05404016 0.11377636 0.07628851]


In [30]:
start = time()
res = de_evp(query, evp_data[0])
end = time() - start

In [29]:
print(f'time: {end}s')
print(f'time/v: {end/D_SIZE * NANO_SECODNS} ns')

print(res)

time: 9.632110595703125e-05s
time/v: 0.09632110595703125 ns
0.8976272586155191
